In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive", force_remount=True)

import subprocess
import sys

packages = [
    "torch torchvision torchaudio torchmetrics",
    "timm transformers rasterio rioxarray albumentations",
    "wandb pandas matplotlib scikit-learn geopandas"
]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + package.split())


import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
import json
import gc

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class PathConfig:
    S1_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/S1Hand"
    S2_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/S2Hand"
    LABEL_HAND = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/LabelHand"
    JRC_WATER = "/content/drive/MyDrive/Sen1Floods11/data/flood_events/HandLabeled/JRCWaterHand"

    TRAIN_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_handlabeled/flood_train_data.csv"
    VAL_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_handlabeled/flood_valid_data.csv"
    TEST_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_handlabeled/flood_test_data.csv"
    BOLIVIA_SPLIT = "/content/drive/MyDrive/Sen1Floods11/splits/flood_handlabeled/flood_handlabeled/flood_bolivia_data.csv"

    METADATA = "/content/drive/MyDrive/Sen1Floods11/metadata/Sen1Floods11_Metadata.geojson"
    OUTPUT_DIR = "/content/drive/MyDrive/Sen1Floods11/EarlyFusionUNet_Outputs"

    def create_dirs(self):
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'checkpoints'), exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_DIR, 'visualizations'), exist_ok=True)

paths = PathConfig()
paths.create_dirs()

def load_and_process_metadata(metadata_path):
    with open(metadata_path, 'r') as f:
        geojson_data = json.load(f)

    features = geojson_data['features']
    metadata_list = []

    for feature in features:
        props = feature['properties']
        region_id = props.get('location', str(props.get('ID', 'unknown')))
        coords = feature['geometry']['coordinates'][0]
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        center_lon = (min(lons) + max(lons)) / 2
        center_lat = (min(lats) + max(lats)) / 2
        props['region_id'] = str(region_id)
        props['center_lon'] = center_lon
        props['center_lat'] = center_lat
        metadata_list.append(props)

    metadata_df = pd.DataFrame(metadata_list)
    print(f"Loaded metadata for {len(metadata_df)} regions")
    return metadata_df

metadata = load_and_process_metadata(paths.METADATA)

def create_file_mapping(base_dir):
    mapping = {}
    if not os.path.exists(base_dir):
        print(f"Warning: Directory does not exist: {base_dir}")
        return mapping

    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.tif'):
                full_path = os.path.join(root, file)
                if file in mapping:
                    raise RuntimeError(f"Duplicate filename detected: {file}")
                mapping[file] = full_path
    return mapping

s1_mapping = create_file_mapping(paths.S1_HAND)
s2_mapping = create_file_mapping(paths.S2_HAND)
label_mapping = create_file_mapping(paths.LABEL_HAND)
jrc_mapping = create_file_mapping(paths.JRC_WATER)

print(f"S1: {len(s1_mapping)} files")
print(f"S2: {len(s2_mapping)} files")
print(f"Labels: {len(label_mapping)} files")
print(f"JRC: {len(jrc_mapping)} files")

def load_and_process_split(split_path, split_name):
    if not os.path.exists(split_path):
        print(f"Split file not found: {split_path}")
        return None

    df = pd.read_csv(split_path)
    cols = list(df.columns)
    s1_col, label_col = cols[0], cols[1]
    df = df.rename(columns={s1_col: 's1_filename', label_col: 'label'})

    def extract_region_from_filename(filename):
        if pd.isna(filename):
            return 'unknown'
        filename_str = str(filename).strip()
        region_mapping = {
            'Bolivia': 'Bolivia', 'Colombia': 'Colombia', 'Ghana': 'Ghana',
            'India': 'India', 'Cambodia': 'Cambodia', 'Mekong': 'Cambodia',
            'Nigeria': 'Nigeria', 'Pakistan': 'Pakistan', 'Paraguay': 'Paraguay',
            'Somalia': 'Somalia', 'Spain': 'Spain', 'Sri-Lanka': 'Sri-Lanka',
            'Sri Lanka': 'Sri-Lanka', 'USA': 'USA'
        }
        for prefix, region in region_mapping.items():
            if filename_str.startswith(prefix):
                return region
        return 'unknown'

    df['region_id'] = df['label'].apply(extract_region_from_filename).astype(str)
    df['s1_filename'] = df['s1_filename'].astype(str)
    df['label'] = df['label'].astype(str)

    print(f"{split_name}: {len(df)} samples, {df['region_id'].nunique()} regions")
    return df

train_df = load_and_process_split(paths.TRAIN_SPLIT, "Training")
val_df = load_and_process_split(paths.VAL_SPLIT, "Validation")
test_df = load_and_process_split(paths.TEST_SPLIT, "Test")
bolivia_df = load_and_process_split(paths.BOLIVIA_SPLIT, "Bolivia")

print(f"\nTraining: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")
print(f"Bolivia: {len(bolivia_df)} samples")


def load_sample_tiff(path):
    try:
        with rasterio.open(path) as src:
            return src.read()
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

def compute_regional_water_statistics(df, split_name):
    print(f"\nComputing water statistics for {split_name}...")
    regional_stats = []

    for region_id in sorted(df['region_id'].unique()):
        region_df = df[df['region_id'] == region_id]
        total_pixels = water_pixels = background_pixels = ignore_pixels = 0
        permanent_water_pixels = flood_pixels = 0

        for _, row in region_df.iterrows():
            label_filename = row['label']
            label_path = label_mapping.get(label_filename)
            jrc_filename = label_filename.replace('LabelHand', 'JRCWaterHand')
            jrc_path = jrc_mapping.get(jrc_filename)

            if not label_path:
                continue

            label = load_sample_tiff(label_path)
            if label is None:
                continue
            label = label[0]

            unique_vals = np.unique(label)
            valid_vals = {-1, 0, 1, 255}
            if not all(v in valid_vals for v in unique_vals):
                mask_invalid = ~np.isin(label, [-1, 0, 1, 255])
                label = np.where(mask_invalid, 255, label)

            total_pixels += label.size
            water_pixels += (label == 1).sum()
            background_pixels += (label == 0).sum()
            ignore_pixels += ((label == -1) | (label == 255)).sum()

            if jrc_path:
                jrc = load_sample_tiff(jrc_path)
                if jrc is not None:
                    jrc = jrc[0]
                    permanent_water_pixels += ((jrc == 1) & (label == 1)).sum()
                    flood_pixels += ((jrc == 0) & (label == 1)).sum()

        valid_pixels = max(1, total_pixels - ignore_pixels)
        regional_stats.append({
            'region_id': region_id,
            'split': split_name,
            'total_samples': len(region_df),
            'total_pixels': total_pixels,
            'valid_pixels': valid_pixels,
            'water_pixels': water_pixels,
            'background_pixels': background_pixels,
            'permanent_water_pixels': permanent_water_pixels,
            'flood_pixels': flood_pixels,
            'water_ratio': water_pixels / valid_pixels,
            'flood_ratio': flood_pixels / valid_pixels,
            'permanent_water_ratio': permanent_water_pixels / valid_pixels
        })

    return pd.DataFrame(regional_stats)

train_stats = compute_regional_water_statistics(train_df, 'train')
val_stats = compute_regional_water_statistics(val_df, 'val')
test_stats = compute_regional_water_statistics(test_df, 'test')

all_stats = pd.concat([train_stats, val_stats, test_stats], ignore_index=True)
stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
all_stats.to_csv(stats_path, index=False)
print(f"\nRegional statistics saved to: {stats_path}")


class DoubleConv(nn.Module):
    """Double convolution block for U-Net"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, 2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        # Handle size mismatch
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2,
                                     diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    """Output convolution"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        return self.conv(x)

class EarlyFusionUNet(nn.Module):
    """
    Early Fusion U-Net for flood detection

    Concatenates all input channels (S1 + S2 + NDVI + NDWI = 17 channels)
    and processes them through a standard U-Net architecture.

    Architecture:
    - Encoder: 4 downsampling stages (64, 128, 256, 512)
    - Bottleneck: 1024 channels
    - Decoder: 4 upsampling stages with skip connections
    - Output: 2 classes (background, water)
    """
    def __init__(self, in_channels=17, num_classes=2, bilinear=True):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.bilinear = bilinear

        # Encoder
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)

        # Decoder
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, num_classes)

    def forward(self, x, metadata=None):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder with skip connections
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)

        return logits


class Sen1Floods11Dataset(Dataset):
    """Dataset class for Sen1Floods11 with early fusion"""
    def __init__(self, df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
                 transform=None):
        self.df = df.reset_index(drop=True)
        self.s1_mapping = s1_mapping
        self.s2_mapping = s2_mapping
        self.label_mapping = label_mapping
        self.jrc_mapping = jrc_mapping
        self.transform = transform

    def load_tiff(self, path):
        try:
            with rasterio.open(path) as src:
                data = src.read()
                data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
                return data
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None

    def compute_indices(self, s2_data):
        green = s2_data[2].astype(np.float32)
        red = s2_data[3].astype(np.float32)
        nir = s2_data[7].astype(np.float32)

        eps = 1e-8
        ndvi = (nir - red) / (nir + red + eps)
        ndvi = np.nan_to_num(np.clip(ndvi, -1, 1), nan=0.0)

        ndwi = (green - nir) / (green + nir + eps)
        ndwi = np.nan_to_num(np.clip(ndwi, -1, 1), nan=0.0)

        return ndvi, ndwi

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s1_filename = row.iloc[0]
        label_filename = row.iloc[1]
        s2_filename = label_filename.replace('LabelHand', 'S2Hand')

        s1_path = self.s1_mapping.get(s1_filename)
        s2_path = self.s2_mapping.get(s2_filename)
        label_path = self.label_mapping.get(label_filename)

        s1 = self.load_tiff(s1_path)
        s2 = self.load_tiff(s2_path)
        label = self.load_tiff(label_path)

        if label is None:
            raise ValueError(f"Could not load label for {label_filename}")
        label = label[0].astype(np.int64)

        mask_invalid = ~np.isin(label, [-1, 0, 1, 255])
        if mask_invalid.any():
            label = np.where(mask_invalid, 255, label)
        label = np.where(label == -1, 255, label)

        channels = []

        # Add S1 channels (VV, VH)
        if s1 is not None:
            s1_vv = s1[0].astype(np.float32)
            s1_vh = s1[1].astype(np.float32)

            s1_vv = np.nan_to_num(s1_vv, nan=-25.0, posinf=0.0, neginf=-50.0)
            s1_vh = np.nan_to_num(s1_vh, nan=-25.0, posinf=0.0, neginf=-50.0)
            s1_vv = np.clip(s1_vv, -50, 0)
            s1_vh = np.clip(s1_vh, -50, 0)
            s1_vv = (s1_vv + 50) / 50.0
            s1_vh = (s1_vh + 50) / 50.0

            channels.extend([s1_vv, s1_vh])

        # Add S2 channels
        if s2 is not None:
            s2f = s2.astype(np.float32)
            s2f = np.nan_to_num(s2f, nan=0.0, posinf=10000.0, neginf=0.0)
            s2n = np.clip(s2f / 10000.0, 0, 1)

            for i in range(s2n.shape[0]):
                channels.append(s2n[i])

            # Add spectral indices
            ndvi, ndwi = self.compute_indices(s2f)
            channels.extend([ndvi, ndwi])

        if len(channels) == 0:
            raise ValueError("No input channels available")

        image = np.stack(channels, axis=-1)

        if np.isnan(image).any() or np.isinf(image).any():
            image = np.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)

        if self.transform:
            augmented = self.transform(image=image, mask=label)
            image = augmented['image']
            label = augmented['mask']

            if not isinstance(label, torch.Tensor):
                label = torch.from_numpy(label).long()
            else:
                label = label.long()
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float()
            label = torch.from_numpy(label).long()

        if torch.isnan(image).any() or torch.isinf(image).any():
            image = torch.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)
        if label.min() < 0 or label.max() > 255:
            label = torch.clamp(label, 0, 255)

        region_id = str(label_filename).split('_')[0]
        metadata = {'region_id': region_id, 'filename': label_filename}

        return image, label, metadata

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1),
                rotate=(-15, 15), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ElasticTransform(alpha=1, sigma=50, p=0.2),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([ToTensorV2()])

def create_dataloaders(train_df, val_df, test_df, bolivia_df, batch_size=4):
    train_dataset = Sen1Floods11Dataset(
        train_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_train_transforms()
    )
    val_dataset = Sen1Floods11Dataset(
        val_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_val_transforms()
    )
    test_dataset = Sen1Floods11Dataset(
        test_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_val_transforms()
    )
    bolivia_dataset = Sen1Floods11Dataset(
        bolivia_df, s1_mapping, s2_mapping, label_mapping, jrc_mapping,
        get_val_transforms()
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                             shuffle=True, num_workers=2,
                             pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                           shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
    bolivia_loader = DataLoader(bolivia_dataset, batch_size=batch_size,
                               shuffle=False, num_workers=2, pin_memory=True)

    print(f"Dataloaders created:")
    print(f"  Train: {len(train_dataset)} samples")
    print(f"  Val: {len(val_dataset)} samples")
    print(f"  Test: {len(test_dataset)} samples")
    print(f"  Bolivia: {len(bolivia_dataset)} samples")

    return train_loader, val_loader, test_loader, bolivia_loader


class SegmentationMetrics:
    def __init__(self, num_classes=2, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index

    @torch.no_grad()
    def iou_per_class(self, logits, target):
        pred = logits.argmax(dim=1)
        mask = (target != self.ignore_index)
        ious = []

        for c in range(self.num_classes):
            pred_c = (pred == c) & mask
            targ_c = (target == c) & mask
            inter = (pred_c & targ_c).sum().item()
            union = (pred_c | targ_c).sum().item()
            iou = inter / (union + 1e-8)
            ious.append(iou)

        return ious

class HybridLoss(nn.Module):
    def __init__(self, num_classes=2, focal_alpha=0.75, focal_gamma=2.0,
                 dice_weight=0.6, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.dice_weight = dice_weight
        self.ce_weight = 1.0 - dice_weight
        self.ignore_index = ignore_index

        self.class_weights = torch.tensor(
            [1.0 - focal_alpha, focal_alpha],
            dtype=torch.float32
        )

    def forward(self, logits, target, sample_weights=None):
        B, C, H, W = logits.shape
        device = logits.device
        class_weights = self.class_weights.to(device)
        mask = (target != self.ignore_index)

        ce = nn.functional.cross_entropy(
            logits, target,
            weight=class_weights,
            ignore_index=self.ignore_index,
            reduction='none'
        )
        ce = torch.where(mask, ce, torch.zeros_like(ce))

        probs = nn.functional.softmax(logits, dim=1)
        tgt = torch.clamp(target, 0, C - 1)
        pt = probs.gather(1, tgt.unsqueeze(1)).squeeze(1).clamp_(1e-6, 1 - 1e-6)
        focal = (1 - pt) ** self.focal_gamma
        focal_ce = focal * ce

        mask_f = mask.float()
        dice_losses = []
        eps = 1e-6

        for b in range(B):
            if mask_f[b].sum() < 1:
                continue

            dice_c = 0.0
            for c in range(C):
                p = probs[b, c] * mask_f[b]
                t = (tgt[b] == c).float() * mask_f[b]
                inter = (p * t).sum()
                denom = p.sum() + t.sum()
                dice_c += 1 - (2 * inter + eps) / (denom + eps)

            dice_c = dice_c / C
            dice_losses.append(dice_c)

        if len(dice_losses) == 0:
            dice_loss = torch.tensor(0.0, device=device)
        else:
            dice_loss = torch.stack(dice_losses).mean()

        focal_per_sample = []
        for b in range(B):
            denom = mask_f[b].sum()
            if denom < 1:
                focal_per_sample.append(torch.tensor(0.0, device=device))
            else:
                focal_per_sample.append(focal_ce[b].sum() / denom)

        focal_per_sample = torch.stack(focal_per_sample)

        if sample_weights is not None:
            w = sample_weights.to(device).float()
            w = w / (w.mean() + 1e-6)
            w = torch.clamp(w, 0.5, 2.5)
            focal_term = (w * focal_per_sample).mean()
        else:
            focal_term = focal_per_sample.mean()

        total = self.dice_weight * dice_loss + self.ce_weight * focal_term
        total = torch.nan_to_num(total, nan=0.0, posinf=1e3, neginf=1e3)

        parts = {
            'dice': dice_loss.detach(),
            'focal': focal_term.detach()
        }

        return total, parts

def compute_region_weights(stats_df):
    train_stats = stats_df[stats_df['split'] == 'train']
    max_ratio = train_stats['water_ratio'].max()
    min_ratio = train_stats['water_ratio'].min()

    weights = {}
    for _, row in train_stats.iterrows():
        ratio = row['water_ratio']
        weight = 1.0 + 2.0 * (ratio - min_ratio) / (max_ratio - min_ratio + 1e-8)
        weights[row['region_id']] = weight

    return weights


def train_epoch(model, loader, loss_fn, optimizer, scheduler, metrics, config, epoch):
    model.train()
    device = config['device']
    scaler = config.get('scaler', None)
    use_amp = bool(config.get('amp', True) and scaler is not None)

    running_loss = 0.0
    valid_batches = 0
    skipped_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for batch_idx, (data, target, meta) in enumerate(pbar):
        data = data.to(device, non_blocking=True).float()
        target = target.to(device, non_blocking=True).long()

        sample_weights = None
        if config.get('use_region_weights', False) and 'region_weights' in config:
            if isinstance(meta, dict) and 'region_id' in meta:
                ids = meta['region_id']
                weights = [config['region_weights'].get(str(ids[i]), 1.0)
                          for i in range(data.size(0))]
                sample_weights = torch.tensor(weights, device=device, dtype=torch.float32)

        optimizer.zero_grad(set_to_none=True)

        try:
            with autocast(enabled=use_amp):
                logits = model(data)
                if not torch.isfinite(logits).all():
                    raise FloatingPointError("Non-finite logits")
                loss, parts = loss_fn(logits, target, sample_weights)

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss")

            if use_amp:
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=config.get('gradient_clip', 10.0)
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=config.get('gradient_clip', 10.0)
                )
                optimizer.step()

            running_loss += loss.item()
            valid_batches += 1
            pbar.set_postfix({'loss': f"{running_loss/max(1,valid_batches):.4f}"})

        except FloatingPointError:
            skipped_batches += 1
            continue

    return {
        'epoch': epoch,
        'train/loss': running_loss / max(1, valid_batches),
        'train/valid_batches': valid_batches,
        'train/skipped_batches': skipped_batches,
    }

def validate_epoch(model, loader, loss_fn, metrics, config, epoch, split_name="Val"):
    model.eval()
    device = config['device']
    scaler = config.get('scaler', None)
    use_amp = bool(config.get('amp', True) and scaler is not None)

    running_loss = 0.0
    valid_batches = 0
    skipped_batches = 0
    region_ious = defaultdict(list)
    all_ious = []

    with torch.no_grad():
        for batch_idx, (data, target, meta) in enumerate(
            tqdm(loader, desc=f"Epoch {epoch} [{split_name}]")
        ):
            data = data.to(device, non_blocking=True).float()
            target = target.to(device, non_blocking=True).long()

            try:
                with autocast(enabled=use_amp):
                    logits = model(data)
                    loss, _ = loss_fn(logits, target, None)

                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite val loss")

                running_loss += loss.item()
                valid_batches += 1

                ious = metrics.iou_per_class(logits, target)
                all_ious.append(ious)

                regions = meta['region_id'] if isinstance(meta, dict) else \
                         [m['region_id'] for m in meta]
                for rid in regions:
                    region_ious[str(rid)].append(ious)

            except FloatingPointError:
                skipped_batches += 1
                continue

    if valid_batches == 0:
        return {
            'epoch': epoch,
            f'{split_name.lower()}/loss': 999.0,
            f'{split_name.lower()}/mIoU': 0.0,
            f'{split_name.lower()}/water_IoU': 0.0,
            f'{split_name.lower()}/bg_IoU': 0.0,
        }, torch.zeros(config['num_classes']), {}

    avg_loss = running_loss / valid_batches
    epoch_iou = np.mean(all_ious, axis=0) if all_ious else \
                np.zeros(config['num_classes'])

    per_region_iou = {}
    for region_id, iou_list in region_ious.items():
        if len(iou_list) > 0:
            region_iou = np.mean(iou_list, axis=0)
            per_region_iou[region_id] = {
                'mean_IoU': region_iou.mean(),
                'water_IoU': region_iou[1],
                'bg_IoU': region_iou[0],
                'samples': len(iou_list)
            }

    return {
        'epoch': epoch,
        f'{split_name.lower()}/loss': avg_loss,
        f'{split_name.lower()}/mIoU': float(epoch_iou.mean()),
        f'{split_name.lower()}/water_IoU': float(epoch_iou[1]),
        f'{split_name.lower()}/bg_IoU': float(epoch_iou[0]),
    }, epoch_iou, per_region_iou


TRAIN_CONFIG = {
    'num_classes': 2,
    'in_channels': 17,  # S1(2) + S2(13) + NDVI(1) + NDWI(1)
    'bilinear': True,

    # Training hyperparameters (same as original)
    'epochs': 150,
    'batch_size': 8,
    'learning_rate': 5e-5,
    'weight_decay': 1e-4,
    'min_lr': 1e-7,

    # Loss configuration
    'focal_alpha': 0.75,
    'focal_gamma': 2.0,
    'dice_weight': 0.6,

    # Optimizer and scheduler
    'lr_scheduler': 'cosine',
    'warmup_epochs': 10,

    # Training stability
    'gradient_clip': 10.0,
    'amp': True,

    # Region weights
    'use_region_weights': True,

    # Validation and checkpointing
    'val_freq': 1,
    'save_freq': 10,
    'early_stopping_patience': 30,

    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

TRAIN_CONFIG['scaler'] = GradScaler(enabled=TRAIN_CONFIG['amp'])


#  Main Training Loop

print("TRAINING")

# Create dataloaders
train_loader, val_loader, test_loader, bolivia_loader = create_dataloaders(
    train_df, val_df, test_df, bolivia_df,
    batch_size=TRAIN_CONFIG['batch_size']
)

# Create model
model = EarlyFusionUNet(
    in_channels=TRAIN_CONFIG['in_channels'],
    num_classes=TRAIN_CONFIG['num_classes'],
    bilinear=TRAIN_CONFIG['bilinear']
).to(TRAIN_CONFIG['device'])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Loss and metrics
loss_fn = HybridLoss(
    num_classes=TRAIN_CONFIG['num_classes'],
    focal_alpha=TRAIN_CONFIG['focal_alpha'],
    focal_gamma=TRAIN_CONFIG['focal_gamma'],
    dice_weight=TRAIN_CONFIG['dice_weight']
).to(TRAIN_CONFIG['device'])

metrics = SegmentationMetrics(num_classes=TRAIN_CONFIG['num_classes'])

# Optimizer and scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG['learning_rate'],
    weight_decay=TRAIN_CONFIG['weight_decay']
)

warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    total_iters=TRAIN_CONFIG['warmup_epochs']
)

main_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRAIN_CONFIG['epochs'] - TRAIN_CONFIG['warmup_epochs'],
    eta_min=TRAIN_CONFIG['min_lr']
)

scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    [warmup_scheduler, main_scheduler],
    milestones=[TRAIN_CONFIG['warmup_epochs']]
)

# Compute region weights
stats_path = os.path.join(paths.OUTPUT_DIR, 'region_water_stats.csv')
region_stats_df = pd.read_csv(stats_path)
TRAIN_CONFIG['region_weights'] = compute_region_weights(region_stats_df)

print("\nRegion weights:")
for region_id, weight in sorted(TRAIN_CONFIG['region_weights'].items(),
                                key=lambda x: x[1], reverse=True):
    print(f"  {region_id}: {weight:.4f}")

# Training loop
best_val_iou = 0.0
patience_counter = 0
train_history = []
val_history = []

print(f"\nStarting training for {TRAIN_CONFIG['epochs']} epochs...")

for epoch in range(1, TRAIN_CONFIG['epochs'] + 1):
    # Train
    train_log = train_epoch(model, train_loader, loss_fn, optimizer,
                           scheduler, metrics, TRAIN_CONFIG, epoch)
    train_history.append(train_log)

    # Validate
    if epoch % TRAIN_CONFIG['val_freq'] == 0:
        val_log, val_iou, per_region_iou = validate_epoch(
            model, val_loader, loss_fn, metrics, TRAIN_CONFIG, epoch, "Val"
        )
        val_history.append(val_log)

        print(f"\nEpoch {epoch}/{TRAIN_CONFIG['epochs']}")
        print(f"  Train Loss: {train_log['train/loss']:.4f}")
        print(f"  Val Loss: {val_log['val/loss']:.4f} | "
              f"mIoU: {val_log['val/mIoU']:.4f} | "
              f"Water IoU: {val_log['val/water_IoU']:.4f}")

        # Save best model
        if val_log['val/water_IoU'] > best_val_iou:
            best_val_iou = val_log['val/water_IoU']
            patience_counter = 0

            checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints',
                                          'early_fusion_unet_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_iou': val_iou,
                'config': TRAIN_CONFIG
            }, checkpoint_path)
            print(f"  Saved best model (Water IoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= TRAIN_CONFIG['early_stopping_patience']:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    # Step scheduler
    scheduler.step()

    # Save periodic checkpoint
    if epoch % TRAIN_CONFIG['save_freq'] == 0:
        checkpoint_path = os.path.join(paths.OUTPUT_DIR, 'checkpoints',
                                      f'early_fusion_unet_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': TRAIN_CONFIG
        }, checkpoint_path)

# Save training history
history_path = os.path.join(paths.OUTPUT_DIR, 'early_fusion_unet_history.json')
with open(history_path, 'w') as f:
    json.dump({'train': train_history, 'val': val_history}, f, indent=2)

print(f" COMPLETE")
print(f"Best Validation Water IoU: {best_val_iou:.4f}")




In [ ]:

print("TEST SET EVALUATION")


# Load best model
checkpoint = torch.load(os.path.join(paths.OUTPUT_DIR, 'checkpoints',
                                     'early_fusion_unet_best.pth'),
                       weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate on test set
test_log, test_iou, test_per_region = validate_epoch(
    model, test_loader, loss_fn, metrics, TRAIN_CONFIG,
    checkpoint['epoch'], "Test"
)

print(f"\nTest Set Results:")
print(f"  Loss: {test_log['test/loss']:.4f}")
print(f"  mIoU: {test_log['test/mIoU']:.4f}")
print(f"  Water IoU: {test_log['test/water_IoU']:.4f}")
print(f"  Background IoU: {test_log['test/bg_IoU']:.4f}")

print(f"\nPer-Region Test IoU:")
for region_id, stats in sorted(test_per_region.items(),
                               key=lambda x: x[1]['water_IoU'], reverse=True):
    print(f"  {region_id}: Water={stats['water_IoU']:.4f}, "
          f"Mean={stats['mean_IoU']:.4f}, Samples={stats['samples']}")

# Bolivia Set Evaluation

print("\n" + "="*80)
print("BOLIVIA SET EVALUATION")
print("="*80)

# Evaluate on Bolivia set
bolivia_log, bolivia_iou, bolivia_per_region = validate_epoch(
    model, bolivia_loader, loss_fn, metrics, TRAIN_CONFIG,
    checkpoint['epoch'], "Bolivia"
)

print(f"\nBolivia Set Results:")
print(f"  Loss: {bolivia_log['bolivia/loss']:.4f}")
print(f"  mIoU: {bolivia_log['bolivia/mIoU']:.4f}")
print(f"  Water IoU: {bolivia_log['bolivia/water_IoU']:.4f}")
print(f"  Background IoU: {bolivia_log['bolivia/bg_IoU']:.4f}")

print(f"\nPer-Region Bolivia IoU:")
for region_id, stats in sorted(bolivia_per_region.items(),
                               key=lambda x: x[1]['water_IoU'], reverse=True):
    print(f"  {region_id}: Water={stats['water_IoU']:.4f}, "
          f"Mean={stats['mean_IoU']:.4f}, Samples={stats['samples']}")


TEST SET EVALUATION


Epoch 25 [Test]:   0%|          | 0/12 [00:00<?, ?it/s]/tmp/ipython-input-840235861.py:763: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
Epoch 25 [Test]: 100%|██████████| 12/12 [00:19<00:00,  1.61s/it]



Test Set Results:
  Loss: 0.2074
  mIoU: 0.8810
  Water IoU: 0.7871
  Background IoU: 0.9749

Per-Region Test IoU:
  Mekong: Water=0.9109, Mean=0.9490, Samples=6
  India: Water=0.8909, Mean=0.9325, Samples=14
  Sri-Lanka: Water=0.8754, Mean=0.9201, Samples=9
  Spain: Water=0.8482, Mean=0.9066, Samples=6
  Nigeria: Water=0.8410, Mean=0.9015, Samples=4
  Somalia: Water=0.8207, Mean=0.8979, Samples=6
  Paraguay: Water=0.8056, Mean=0.8953, Samples=14
  Pakistan: Water=0.7712, Mean=0.8540, Samples=6
  Ghana: Water=0.7322, Mean=0.8465, Samples=10
  USA: Water=0.7296, Mean=0.8605, Samples=14

BOLIVIA SET EVALUATION


Epoch 25 [Bolivia]: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]


Bolivia Set Results:
  Loss: 0.1778
  mIoU: 0.8889
  Water IoU: 0.8123
  Background IoU: 0.9655

Per-Region Bolivia IoU:
  Bolivia: Water=0.8138, Mean=0.8898, Samples=14


In [ ]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")


Model Parameters:
  Total: 13,399,490
  Trainable: 13,399,490
